# 🏭 AutoFactoryScope: YOLO11 Training Pipeline

**Objective**: Train YOLO11 on factory layout robot detection dataset.

**Environment**: Google Colab (free T4 GPU) or local machine with CUDA.

**Runtime**: ~30-60 minutes for 50 epochs on 138 images.

## 📋 Setup

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install Ultralytics (YOLO11)
!pip install ultralytics -q

# Verify installation
from ultralytics import YOLO
import torch

print(f"Ultralytics version: {YOLO.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 📦 Upload Dataset

**Option 1**: Upload ZIP to Colab

1. Compress dataset locally: `datasets/v2_roboflow_export` → `v2_dataset.zip`
2. Upload to Colab using file upload widget

**Option 2**: Mount Google Drive

1. Upload `v2_dataset.zip` to Drive
2. Run cell below to mount Drive

In [ ]:
# Option 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Unzip dataset
!unzip -q /content/drive/MyDrive/AutoFactoryScope/v2_dataset.zip -d /content/dataset

In [ ]:
# Verify dataset structure
!ls -la /content/dataset
!cat /content/dataset/data.yaml

## 🔧 Fix data.yaml Paths

Roboflow exports use relative paths (`../train/images`). We need absolute paths for Colab.

In [ ]:
import yaml
from pathlib import Path

# Read original data.yaml
data_yaml_path = Path('/content/dataset/data.yaml')
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

# Update paths to absolute
base_path = Path('/content/dataset')
data_config['train'] = str(base_path / 'train' / 'images')
data_config['val'] = str(base_path / 'valid' / 'images')
data_config['test'] = str(base_path / 'test' / 'images')

# Save updated config
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print("Updated data.yaml:")
print(yaml.dump(data_config, default_flow_style=False))

## 🚀 Training

In [ ]:
# Initialize YOLO11 nano model (fastest, smallest)
model = YOLO('yolo11n.pt')  # Download pretrained weights

# Train
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    imgsz=512,  # Match your tile size
    batch=16,   # Adjust based on GPU memory (8-32)
    
    # Augmentation (important for small datasets)
    hsv_h=0.015,  # Hue variation
    hsv_s=0.7,    # Saturation variation
    hsv_v=0.4,    # Value variation
    degrees=10.0, # Rotation (factory layouts may be rotated)
    translate=0.1,
    scale=0.5,
    shear=0.0,
    flipud=0.5,   # Flip up-down (layouts can be inverted)
    fliplr=0.5,   # Flip left-right
    mosaic=1.0,   # Mosaic augmentation (very effective)
    mixup=0.0,    # Keep 0 for detection tasks
    
    # Optimization
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    
    # Misc
    patience=10,  # Early stopping
    save=True,
    save_period=10,  # Save every 10 epochs
    cache=True,   # Cache images for faster training
    device=0,     # GPU 0 (use 'cpu' if no GPU)
    workers=4,
    project='runs/train',
    name='robot_detector_v1',
    exist_ok=True,
    pretrained=True,
    verbose=True
)

## 📊 Evaluate Results

In [ ]:
# Load best model
best_model = YOLO('runs/train/robot_detector_v1/weights/best.pt')

# Validate on test set
metrics = best_model.val(data='/content/dataset/data.yaml')

print(f"\nMetrics:")
print(f"  mAP50: {metrics.box.map50:.3f}")
print(f"  mAP50-95: {metrics.box.map:.3f}")
print(f"  Precision: {metrics.box.p.mean():.3f}")
print(f"  Recall: {metrics.box.r.mean():.3f}")

In [ ]:
# Visualize predictions on test set
from IPython.display import Image, display

# Run inference on test images
test_results = best_model.predict(
    source='/content/dataset/test/images',
    conf=0.25,
    save=True,
    project='runs/predict',
    name='test_predictions'
)

# Display first 5 predictions
import glob
pred_images = sorted(glob.glob('runs/predict/test_predictions/*.jpg'))[:5]
for img_path in pred_images:
    print(f"\n{Path(img_path).name}")
    display(Image(filename=img_path, width=600))

## 📥 Download Model

Download `best.pt` to your local machine for ONNX export.

In [ ]:
# Copy to Google Drive (if mounted)
!mkdir -p /content/drive/MyDrive/AutoFactoryScope/models
!cp runs/train/robot_detector_v1/weights/best.pt /content/drive/MyDrive/AutoFactoryScope/models/

print("Model saved to Google Drive: MyDrive/AutoFactoryScope/models/best.pt")
print("Download manually or sync with Drive desktop app.")

In [ ]:
# Alternative: Direct download (Colab file browser)
from google.colab import files
files.download('runs/train/robot_detector_v1/weights/best.pt')

## ⏭️ Next Steps

1. Download `best.pt` to `c:\Users\georgem\source\repos\AutoFactoryScope\models\`
2. Open `02_export_onnx.ipynb` to convert to ONNX format
3. Run `validate_onnx_parity.py` to ensure accuracy is preserved